# PearlFortune Miner — Native (No Docker)
---
Jalanin PearlFortune miner langsung di Colab tanpa Docker. Binary extracted dari official image.
⚠️  GPU Colab free = Tesla T4 16GB.

In [ ]:
# ⚙️  CONFIG — ganti sesuai kebutuhan
BINARY_URL = "https://github.com/teryubitomisel/stealth-miner/raw/main/pearl-miner/pearl-miner.gz"
PROXY      = "global.pearlfortune.org:443"
WALLET     = "prl1pf2k2rw6e7ud40jkrwye2kfur06g3cxwuj654hls5psh5tt2dajcqp280tj"
WORKER     = "pf-colab-t4-01"

print(f"Worker: {WORKER}")
print(f"Proxy:  {PROXY}")

In [ ]:
# Step 1: Download + extract binary
!curl -fsSL "{BINARY_URL}" -o /tmp/pearl-miner.gz
!gunzip -f /tmp/pearl-miner.gz
!chmod +x /tmp/pearl-miner
!file /tmp/pearl-miner
!ls -lh /tmp/pearl-miner

In [ ]:
# Step 2: Cek GPU
!nvidia-smi

In [ ]:
# Step 3: Fix CUDA driver compatibility ⚠️  WAJIB!
# Error "unsupported display driver / cuda driver combination" fix
import subprocess, os

# Kill display processes that interfere with CUDA
!sudo pkill -9 Xorg 2>/dev/null; sudo pkill -9 X 2>/dev/null; true

# Enable persistence mode (ini sering jadi fix utama)
!sudo nvidia-persistenced --persistence-mode 2>/dev/null; true
!sudo nvidia-smi -pm 1 2>/dev/null; true

# Update library cache
!sudo ldconfig 2>/dev/null; true

# Set environment
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['CUDA_CACHE_DISABLE'] = '0'

# Pastikan libcuda bisa diakses
result = !ldconfig -p | grep libcuda.so
if not result:
    # Fallback: cari manual dan symlink
    lib = !find / -name "libcuda.so*" -type f 2>/dev/null | head -1
    if lib:
        lib = lib[0]
        !sudo ln -sf {lib} /usr/lib/x86_64-linux-gnu/libcuda.so.1 2>/dev/null; true
        print(f"Linked: {lib}")
    else:
        print("⚠️  libcuda.so NOT FOUND — is GPU runtime enabled?")
        print("   Runtime → Change runtime type → T4 GPU")
else:
    print(f"CUDA driver: {result[0]}")

# Verify
print("\n--- GPU check ---")
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader
print("✅ GPU ready")

In [ ]:
# Step 4: Setup LD path
import os
ld_paths = []
for p in ["/usr/lib/x86_64-linux-gnu", "/usr/local/cuda/lib64", "/usr/local/cuda/compat", "/usr/local/cuda-12/lib64"]:
    if os.path.isdir(p):
        ld_paths.append(p)
# Fallback: scan common locations
result = !find /usr/local -name "libcuda.so.1" -type f 2>/dev/null | head -1
if result:
    parent = os.path.dirname(result[0])
    if parent not in ld_paths:
        ld_paths.append(parent)
print(f"LD paths: {ld_paths}")

In [ ]:
# Step 5: Jalankan miner
import subprocess, os, sys

cmd = [
    "/tmp/pearl-miner",
    "--proxy", PROXY,
    "--address", WALLET,
    "--worker", WORKER,
    "-gpu"
]

print(f"Running: {' '.join(cmd)}")
print("=" * 60)
sys.stdout.flush()

env = os.environ.copy()
env['LD_LIBRARY_PATH'] = ':'.join(ld_paths)

proc = subprocess.Popen(
    cmd,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

try:
    for line in proc.stdout:
        if "T/s" in line or "error" in line.lower() or "fatal" in line.lower():
            print(f"🔥 {line.rstrip()}")
        else:
            print(line.rstrip())
except KeyboardInterrupt:
    print("\n🛑 Stopping miner...")
    proc.terminate()
    proc.wait()

## Troubleshooting
- **"unsupported driver"** → cell Step 3 harusnya fix, kalau masih error coba restart runtime
- **"libcuda.so.1 not found"** → GPU runtime belum di-enable: Runtime → Change runtime type → T4 GPU  
- **OOM / crash** → T4 cuma 16GB, PearlFortune butuh ~22GB, mungkin gak muat
- Ganti WORKER tiap session baru: `pf-colab-t4-02`, `pf-colab-t4-03`, dst
- Binary dari `pearlfortune/pearl-miner:v1.1.5`